In [ ]:
from __future__ import annotations
# import np
import numpy as np
import os
from getpass import getpass
from typing import Any
from urllib.parse import urljoin
import wrds
import pandas as pd
import requests
import matplotlib.pyplot as plt

# find repo root git
def find_repo_root(path: str) -> str:
    if os.path.isdir(os.path.join(path, '.git')):
        return path
    parent = os.path.dirname(path)
    if parent == path:
        raise FileNotFoundError("No .git directory found in any parent directories.")
    return find_repo_root(parent)

repo_root = find_repo_root(os.getcwd())
print(f"Repo root found at: {repo_root}")


# set up data (processed) 
data_path = os.path.join(repo_root, 'data', 'raw', 'wrds')
external_data_path = os.path.join(repo_root, 'data', 'external')
processed_data_path = os.path.join(repo_root, 'data', 'processed', 'wrds')
figure_output = os.path.join(repo_root, 'output', 'figures', 'wrds')
table_output = os.path.join(repo_root, 'output', 'tables', 'wrds')
os.makedirs(data_path, exist_ok=True)
os.makedirs(processed_data_path, exist_ok=True)
os.makedirs(figure_output, exist_ok=True)
os.makedirs(table_output, exist_ok=True)


In [ ]:
# sales df
df_sales = pd.read_csv(f'{processed_data_path}/wrds_sales_agg_by_segmentxyear.csv')
# only those not making loss ie sales > 0
df_sales = df_sales[df_sales['sales'] > 0]
df_count = pd.read_csv(f'{processed_data_path}/wrds_naics_code_count_by_segmentxyear.csv')
df_naics = pd.read_csv(f'{processed_data_path}/naics_codes_17_22.csv')

In [ ]:
df_count.columns

In [ ]:
# rename naics_code to naics_code_raw
df_count = df_count.rename(columns={'naics_code': 'naics_code_raw'})

# aggregate to x digit NAICS code
df_count['naics_code'] = (
    df_count['naics_code_raw']
    .astype(str)
    .str.extract(r'(\d{3})', expand=False)
)

df_count_agg_by_naicsx_year = (
    df_count
    .groupby(['naics_code', 'firm_segment_name', 'year'], as_index=False)
    .agg(count=('count', 'sum'))
)

# merge with sales df by firm_segment_name and year
df_sales_count_merged = pd.merge(
    df_count_agg_by_naicsx_year,
    df_sales,
    on=['firm_segment_name', 'year'],
    how='left'
)

# crosswalk naicsh_code with df_naics (naics_code is string, match on 5-digit prefix)
#df_naics_5digit = df_naics.copy()
#df_naics_5digit['naicsh_code'] = df_naics_5digit['naics_code'].astype(str).str[:5]
# keep only 5-digit naics rows to avoid duplicates
#df_naics_5digit = df_naics_5digit[df_naics_5digit['naics_code'].astype(str).str.len() == 5].copy()
#df_naics_5digit = df_naics_5digit.rename(columns={'naics_code': 'naics_code', 'naics_title': 'naics_title'})

#df_sales_count_merged = pd.merge(
#    df_sales_count_merged,
#    df_naics_5digit[['naicsh_code', 'naics_code', 'naics_title']],
#    on='naicsh_code',
#    how='left'
#)

# industry sales by year
df_industry_sales_by_year = (
    df_sales_count_merged
    .groupby(['naics_code', 'year'], as_index=False)
    .agg(sales_industry=('sales', 'sum'), count_industry=('count', 'sum'))
)

# merge industry sales with df_sales_count_merged
df_sales_count_merged = pd.merge(
    df_sales_count_merged,
    df_industry_sales_by_year,
    on=['naics_code', 'year'],
    how='left'
)

# build sales weighted count
df_sales_count_merged['sales_weighted_count'] = (
    df_sales_count_merged['count'] / df_sales_count_merged['count_industry'] * df_sales_count_merged['sales_industry']
)

# only include naicsh_code with more than 5 unique years
naicsx_with_enough_data = (
    df_sales_count_merged
    .groupby('naics_code')
    .agg(unique_segment_years=('year', 'nunique'))
    .reset_index()
    .query('unique_segment_years >= 5')['naics_code']
)

df_sales_count_merged = df_sales_count_merged[df_sales_count_merged['naics_code'].isin(naicsx_with_enough_data)]


In [ ]:
df_sales_count_merged.describe().T

In [ ]:
df_sales_count_merged['year'].unique()

## Specification A. Bayesian segment-allocation core

This notebook's main Bayesian specification treats each firm-segment-year's industry composition as a latent allocation vector with a Dirichlet posterior. Pre-2022 observations are used to construct prior pseudo-mass, and post-2021 observations provide the update.

**Model object.** For segment $s$ belonging to firm $f$, let $\theta_{fs}$ denote the latent vector of industry shares across 3-digit NAICS codes. The notebook constructs a sales-weighted pseudo-count prior,
$$
\theta_{fs} \sim \text{Dirichlet}(\alpha^{\text{prior}}_{fs1}, \ldots, \alpha^{\text{prior}}_{fsK}),
$$
where $\alpha^{\text{prior}}$ is formed from pre-period `sales_weighted_count` mass.

**Update rule.** For a target year $t$, the posterior mass is computed as
$$
\alpha^{\text{post}}_{fskt} = \alpha^{\text{prior}}_{fsk} + y_{fskt},
$$
where $y_{fskt}$ is the year-$t$ sales-weighted evidence mass for industry $k$. Posterior draws from `np.random.dirichlet(alpha_post, size=n_draws)` are then translated into segment-industry sales allocations.

**Interpretation note.** These are pseudo-counts rather than literal counts. The specification should therefore be read as a conjugate-style Bayesian allocation model aligned to revenue exposure, not as a literal multinomial event model.

**Table/Figure note.** Any CR4 or HHI distribution produced below inherits uncertainty only from this Dirichlet allocation step. The results do not yet incorporate additional layers of uncertainty such as measurement error in segment sales or uncertainty in the pseudo-count scaling rule.



In [ ]:
# -----------------------------
# 1. Build prior from pre-COVID
# -----------------------------
pre_years = [2017, 2018, 2019, 2020,2021]

df_prior = (
    df_sales_count_merged[df_sales_count_merged["year"].isin(pre_years)]
    .groupby(["gvkey", "conm", "firm_segment_name", "naics_code"], as_index=False)
    .agg(prior_mass=("sales_weighted_count", "sum"))
)

# smoothing so no zero Dirichlet parameters
SMOOTH = 0.00000000000000000000000000001
df_prior["prior_mass"] = df_prior["prior_mass"] + SMOOTH


# -----------------------------
# 2. One-year posterior helper
# -----------------------------
def cr4_distribution_for_year(df_all, df_prior, target_year, n_draws=1000):
    df_year = df_all[df_all["year"] == target_year].copy()

    # Year-specific evidence mass
    df_evidence = (
        df_year.groupby(
            ["gvkey", "conm", "firm_segment_name", "naics_code", "sales"],
            as_index=False
        )
        .agg(evidence_mass=("sales_weighted_count", "sum"))
    )

    # Merge prior + evidence
    df_post = df_evidence.merge(
        df_prior,
        on=["gvkey", "conm", "firm_segment_name", "naics_code"],
        how="outer"
    )

    # Fill missing values
    df_post["conm"] = df_post["conm_x"].combine_first(df_post["conm_y"]) if "conm_x" in df_post.columns else df_post["conm"]
    if "conm_x" in df_post.columns:
        df_post = df_post.drop(columns=["conm_x", "conm_y"])

    df_post["sales"] = df_post["sales"].fillna(0)
    df_post["evidence_mass"] = df_post["evidence_mass"].fillna(0)
    df_post["prior_mass"] = df_post["prior_mass"].fillna(SMOOTH)

    # posterior alpha
    df_post["alpha_post"] = df_post["prior_mass"] + df_post["evidence_mass"]

    # Build posterior draws segment-by-segment
    results = []

    seg_keys = ["gvkey", "conm", "firm_segment_name"]
    for seg_key, grp in df_post.groupby(seg_keys):
        grp = grp.copy()
        industries = grp["naics_code"].tolist()
        alpha = grp["alpha_post"].to_numpy(dtype=float)
        sales = float(grp["sales"].iloc[0])

        # skip zero-sales segments
        if sales <= 0:
            continue

        # posterior draws of industry shares
        theta_draws = np.random.dirichlet(alpha, size=n_draws)  # shape: (draw, K)

        for j, industry in enumerate(industries):
            seg_ind_sales = sales * theta_draws[:, j]
            results.append(pd.DataFrame({
                "draw": np.arange(n_draws),
                "year": target_year,
                "naics_code": industry,
                "gvkey": seg_key[0],
                "conm": seg_key[1],
                "firm_segment_name": seg_key[2],
                "segment_industry_sales": seg_ind_sales
            }))

    df_draws = pd.concat(results, ignore_index=True)

    # Aggregate to firm x industry x draw
    df_firm_industry = (
        df_draws.groupby(["draw", "year", "naics_code", "gvkey", "conm"], as_index=False)
        .agg(industry_sales=("segment_industry_sales", "sum"))
    )

    # Compute market shares within each draw x year x industry
    df_firm_industry["industry_total_sales"] = df_firm_industry.groupby(
        ["draw", "year", "naics_code"]
    )["industry_sales"].transform("sum")

    df_firm_industry["market_share"] = (
        df_firm_industry["industry_sales"] / df_firm_industry["industry_total_sales"]
    )

    # Rank within draw x year x industry
    df_firm_industry = df_firm_industry.sort_values(
        ["draw", "year", "naics_code", "market_share"],
        ascending=[True, True, True, False]
    )

    df_firm_industry["rank"] = df_firm_industry.groupby(
        ["draw", "year", "naics_code"]
    ).cumcount() + 1

    # CR4 per draw
    df_cr4_draws = (
        df_firm_industry[df_firm_industry["rank"] <= 4]
        .groupby(["draw", "year", "naics_code"], as_index=False)
        .agg(cr4=("market_share", "sum"))
    )

    return df_cr4_draws, df_firm_industry


In [ ]:
post_years = sorted(y for y in df_sales_count_merged["year"].unique() if y > 2021)

all_cr4 = []
for yr in post_years:
    df_cr4_draws, _ = cr4_distribution_for_year(
        df_all=df_sales_count_merged,
        df_prior=df_prior,
        target_year=yr,
        n_draws=500
    ) 
    all_cr4.append(df_cr4_draws)

df_cr4_dist = pd.concat(all_cr4, ignore_index=True)


In [ ]:
df_cr4_dist 

In [ ]:
df_cr4_summary = (
    df_cr4_dist.groupby(["year", "naics_code"], as_index=False)
    .agg(
        cr4_mean=("cr4", "mean"),
        cr4_p10=("cr4", lambda x: np.percentile(x, 10)),
        cr4_p50=("cr4", lambda x: np.percentile(x, 50)),
        cr4_p90=("cr4", lambda x: np.percentile(x, 90)),
    )
)

print(df_cr4_summary.head())


In [ ]:
df_cr4_summary

In [ ]:
import matplotlib.pyplot as plt

industry = "455"  # replace with your actual label

tmp = df_cr4_summary[df_cr4_summary["naics_code"] == industry].sort_values("year")

plt.figure(figsize=(8, 5))
plt.plot(tmp["year"], tmp["cr4_mean"], marker="o", label="Posterior mean CR4")
plt.fill_between(tmp["year"], tmp["cr4_p10"], tmp["cr4_p90"], alpha=0.2, label="80% interval")
plt.title(f"CR4 distribution over time: NAICS {industry}")
plt.xlabel("Year")
plt.ylabel("CR4")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
post_years = sorted(y for y in df_sales_count_merged["year"].unique() if y > 2021)

all_hhi = []
for yr in post_years:
    _, df_firm_industry = cr4_distribution_for_year(
        df_all=df_sales_count_merged,
        df_prior=df_prior,
        target_year=yr,
        n_draws=500
    )

    df_hhi_draws = (
        df_firm_industry.groupby(["draw", "year", "naics_code"], as_index=False)
        .agg(hhi=("market_share", lambda x: (x**2).sum() * 10000))
    )

    all_hhi.append(df_hhi_draws)

df_hhi_dist = pd.concat(all_hhi, ignore_index=True)



In [ ]:
df_hhi_dist 

In [ ]:
df_sales['conm'].nunique()
print(f"Unique companies in sales data: {df_sales['conm'].nunique()}")
print(f"Unique segment in sales data: {df_sales['firm_segment_name'].nunique()}")
print(f"Year Coverage in sales data: {df_sales['year'].min()} - {df_sales['year'].max()}")
# how many sp500 companies in the sales data?
print(f"Number of S&P 500 companies in sales data: {df_sales[df_sales['curr_sp500_flag'] == 1]['conm'].nunique()}")


# unique sp500 companies by year
sp500_companies_by_year = (
    df_sales[df_sales['curr_sp500_flag'] == 1]
    .groupby('year')['conm']
    .nunique()
    .reset_index(name='unique_sp500_companies')
)
print(sp500_companies_by_year)


In [ ]:
df_sales.columns

In [ ]:
df_sales_agg_across_years = (
    df_sales
    .groupby(['conm', 'firm_segment_name', 'gvkey', 'cik', 'curr_sp500_flag'], as_index=False)
    .agg(sales=('sales', 'sum'))
)

df_sales_agg_across_years

In [ ]:
df_count.describe().T

## Specification B. Traditional fixed-assignment benchmark

This block constructs the deterministic benchmark used throughout the notebook.

**Benchmark rule.** Within each firm-segment-year, raw industry counts are normalized into a single fixed share vector,
$$
\hat{s}^{\text{bench}}_{fskt} = \frac{\text{count}_{fskt}}{\sum_j \text{count}_{fsjt}}.
$$
Segment sales are then deterministically allocated as
$$
\widehat{\text{sales}}^{\text{bench}}_{fskt} = \text{segment sales}_{fst} \times \hat{s}^{\text{bench}}_{fskt}.
$$
Aggregating over segments yields firm-industry sales, firm market shares, and then benchmark HHI and CR4 by year and 3-digit NAICS.

**Substantive role.** This is the paper's natural comparison object: a conventional fixed-assignment concentration measure that ignores uncertainty in firm-industry allocation.

**Interpretation note.** Because the benchmark produces one fixed allocation per segment-year, it provides point estimates only. Any gap between this benchmark and the Bayesian posterior summaries below should be interpreted as a consequence of explicitly modeling allocation uncertainty rather than mechanically assigning all ambiguous mass to a single share vector.



In [ ]:
import seaborn as sns

# 3-digit NAICS title map for plotting
df_naics['naics_code_str'] = df_naics['naics_code'].astype(str)
df_naics['naics_code_3digit'] = df_naics['naics_code_str'].str.extract(r'(\d{3})', expand=False)
naics_title_map = (
    df_naics.dropna(subset=['naics_code_3digit'])
    .sort_values(['naics_code_3digit', 'naics_code_str'])
    .drop_duplicates(subset=['naics_code_3digit'])
    [['naics_code_3digit', 'naics_title']]
    .rename(columns={'naics_code_3digit': 'naics_code'})
)

# Traditional benchmark: fixed-share allocation from current-year count shares
benchmark = df_sales_count_merged.copy()
benchmark['segment_total_count'] = benchmark.groupby(
    ['gvkey', 'conm', 'firm_segment_name', 'year']
)['count'].transform('sum')
benchmark['benchmark_share'] = benchmark['count'] / benchmark['segment_total_count']
benchmark['benchmark_segment_industry_sales'] = benchmark['sales'] * benchmark['benchmark_share']

benchmark_firm_industry = (
    benchmark.groupby(['year', 'naics_code', 'gvkey', 'conm'], as_index=False)
    .agg(industry_sales=('benchmark_segment_industry_sales', 'sum'))
)
benchmark_firm_industry['industry_total_sales'] = benchmark_firm_industry.groupby(
    ['year', 'naics_code']
)['industry_sales'].transform('sum')
benchmark_firm_industry['market_share'] = (
    benchmark_firm_industry['industry_sales'] / benchmark_firm_industry['industry_total_sales']
)
benchmark_firm_industry = benchmark_firm_industry.sort_values(
    ['year', 'naics_code', 'market_share'], ascending=[True, True, False]
)
benchmark_firm_industry['rank'] = benchmark_firm_industry.groupby(
    ['year', 'naics_code']
).cumcount() + 1

benchmark_cr4 = (
    benchmark_firm_industry[benchmark_firm_industry['rank'] <= 4]
    .groupby(['year', 'naics_code'], as_index=False)
    .agg(cr4_benchmark=('market_share', 'sum'))
)

benchmark_hhi = (
    benchmark_firm_industry.groupby(['year', 'naics_code'], as_index=False)
    .agg(hhi_benchmark=('market_share', lambda x: (x**2).sum() * 10000))
)

benchmark_cr4 = benchmark_cr4.merge(naics_title_map, on='naics_code', how='left')
benchmark_hhi = benchmark_hhi.merge(naics_title_map, on='naics_code', how='left')

benchmark_hhi.head()

In [ ]:
df_hhi_summary = (
    df_hhi_dist.groupby(['year', 'naics_code'], as_index=False)
    .agg(
        hhi_mean=('hhi', 'mean'),
        hhi_p10=('hhi', lambda x: np.percentile(x, 10)),
        hhi_p50=('hhi', lambda x: np.percentile(x, 50)),
        hhi_p90=('hhi', lambda x: np.percentile(x, 90)),
    )
)

df_cr4_summary = df_cr4_summary.merge(naics_title_map, on='naics_code', how='left')
df_hhi_summary = df_hhi_summary.merge(naics_title_map, on='naics_code', how='left')

compare_hhi = df_hhi_summary.merge(
    benchmark_hhi[['year', 'naics_code', 'hhi_benchmark']],
    on=['year', 'naics_code'],
    how='left'
)

compare_cr4 = df_cr4_summary.merge(
    benchmark_cr4[['year', 'naics_code', 'cr4_benchmark']],
    on=['year', 'naics_code'],
    how='left'
)

compare_hhi.head()

## Specification C. Aggregate fixed-weight concentration trend

This section builds an aggregate time-series from industry-level concentration measures using **fixed all-time industry sales weights**, excluding 2026 because WRDS coverage for 2026 is partial.

**Weight construction.** Let $w_k^{\text{all}}$ denote industry $k$'s share of total benchmark industry sales over 2017--2025. Aggregate concentration is then formed as
$$
C_t^{\text{agg, fixed}} = \sum_k w_k^{\text{all}} C_{kt},
$$
where $C_{kt}$ is either HHI or CR4 for industry $k$ in year $t$.

**Why this matters.** Fixing the weights isolates changes in concentration from changes in industry composition. The resulting series answers: *holding long-run industry importance fixed, how did concentration evolve?*

**Figure note.** In the plots below, the Bayesian line is the posterior mean aggregate concentration, the shaded band is the 10--90 posterior interval, and the benchmark line is the deterministic aggregate point estimate. For CR4, the benchmark trend is chained with a Törnqvist index rather than a simple fixed-weight level average; see the next note in the code for the exact construction.



In [ ]:
# Aggregate time-series comparison using fixed all-time industry sales weights.
# Exclude 2026 from the trend sample because WRDS coverage for 2026 is partial.
trend_end_year = 2025

benchmark_firm_industry_trend = benchmark_firm_industry[
    benchmark_firm_industry['year'] <= trend_end_year
].copy()
benchmark_hhi_trend = benchmark_hhi[benchmark_hhi['year'] <= trend_end_year].copy()
benchmark_cr4_trend = benchmark_cr4[benchmark_cr4['year'] <= trend_end_year].copy()
df_hhi_dist_trend = df_hhi_dist[df_hhi_dist['year'] <= trend_end_year].copy()
df_cr4_dist_trend = df_cr4_dist[df_cr4_dist['year'] <= trend_end_year].copy()

industry_weights = (
    benchmark_firm_industry_trend.groupby('naics_code', as_index=False)
    .agg(industry_sales_all_time=('industry_sales', 'sum'))
)
industry_weights['industry_weight_all_time'] = (
    industry_weights['industry_sales_all_time'] / industry_weights['industry_sales_all_time'].sum()
)
industry_weights = industry_weights.merge(naics_title_map, on='naics_code', how='left')
industry_weights.sort_values('industry_weight_all_time', ascending=False).head(10)



In [ ]:
# Bayesian weighted aggregate distributions by year
agg_hhi_dist = (
    df_hhi_dist_trend.merge(industry_weights[['naics_code', 'industry_weight_all_time']], on='naics_code', how='inner')
    .assign(weighted_hhi=lambda d: d['hhi'] * d['industry_weight_all_time'])
    .groupby(['draw', 'year'], as_index=False)
    .agg(hhi_weighted=('weighted_hhi', 'sum'))
)

agg_cr4_dist = (
    df_cr4_dist_trend.merge(industry_weights[['naics_code', 'industry_weight_all_time']], on='naics_code', how='inner')
    .assign(weighted_cr4=lambda d: d['cr4'] * d['industry_weight_all_time'])
    .groupby(['draw', 'year'], as_index=False)
    .agg(cr4_weighted=('weighted_cr4', 'sum'))
)

# Benchmark weighted aggregate HHI point estimates by year
agg_hhi_benchmark = (
    benchmark_hhi_trend.merge(industry_weights[['naics_code', 'industry_weight_all_time']], on='naics_code', how='inner')
    .assign(weighted_hhi=lambda d: d['hhi_benchmark'] * d['industry_weight_all_time'])
    .groupby('year', as_index=False)
    .agg(hhi_weighted_benchmark=('weighted_hhi', 'sum'))
)

# Benchmark aggregate CR4: use a Törnqvist index over industry-level CR4 growth,
# with year-specific industry sales shares as the Törnqvist weights.
industry_sales_year = (
    benchmark_firm_industry_trend.groupby(['year', 'naics_code'], as_index=False)
    .agg(industry_sales=('industry_total_sales', 'first'))
)
industry_sales_year['sales_share_year'] = (
    industry_sales_year['industry_sales']
    / industry_sales_year.groupby('year')['industry_sales'].transform('sum')
)

cr4_tornqvist = benchmark_cr4_trend.merge(
    industry_sales_year[['year', 'naics_code', 'sales_share_year']],
    on=['year', 'naics_code'],
    how='inner'
)

cr4_tornqvist = cr4_tornqvist.sort_values(['naics_code', 'year']).copy()
cr4_tornqvist['log_cr4_benchmark'] = np.log(cr4_tornqvist['cr4_benchmark'])
cr4_tornqvist['lag_year'] = cr4_tornqvist.groupby('naics_code')['year'].shift(1)
cr4_tornqvist['lag_log_cr4_benchmark'] = cr4_tornqvist.groupby('naics_code')['log_cr4_benchmark'].shift(1)
cr4_tornqvist['lag_sales_share_year'] = cr4_tornqvist.groupby('naics_code')['sales_share_year'].shift(1)
cr4_tornqvist['log_growth_component'] = (
    0.5 * (cr4_tornqvist['sales_share_year'] + cr4_tornqvist['lag_sales_share_year'])
    * (cr4_tornqvist['log_cr4_benchmark'] - cr4_tornqvist['lag_log_cr4_benchmark'])
)

cr4_tornqvist_step = (
    cr4_tornqvist.dropna(subset=['lag_year', 'log_growth_component'])
    .assign(expected_year=lambda d: d['lag_year'] + 1)
)
cr4_tornqvist_step = cr4_tornqvist_step[
    cr4_tornqvist_step['year'] == cr4_tornqvist_step['expected_year']
].copy()

cr4_tornqvist_step = (
    cr4_tornqvist_step.groupby('year', as_index=False)
    .agg(delta_log_cr4_weighted_benchmark=('log_growth_component', 'sum'))
)

agg_cr4_benchmark_base = (
    benchmark_cr4_trend.merge(industry_weights[['naics_code', 'industry_weight_all_time']], on='naics_code', how='inner')
    .assign(weighted_cr4=lambda d: d['cr4_benchmark'] * d['industry_weight_all_time'])
    .groupby('year', as_index=False)
    .agg(cr4_weighted_benchmark_base=('weighted_cr4', 'sum'))
)

base_year_agg = int(min(agg_hhi_dist['year'].min(), agg_cr4_dist['year'].min()))
all_trend_years = sorted(set(agg_hhi_dist['year']).intersection(set(agg_cr4_dist['year'])))

agg_cr4_benchmark = pd.DataFrame({'year': all_trend_years})
agg_cr4_benchmark = agg_cr4_benchmark.merge(
    cr4_tornqvist_step,
    on='year',
    how='left'
)
agg_cr4_benchmark['delta_log_cr4_weighted_benchmark'] = (
    agg_cr4_benchmark['delta_log_cr4_weighted_benchmark'].fillna(0.0)
)
agg_cr4_benchmark = agg_cr4_benchmark.sort_values('year').copy()
agg_cr4_benchmark['tornqvist_log_index_cr4'] = agg_cr4_benchmark['delta_log_cr4_weighted_benchmark'].cumsum()
agg_cr4_benchmark['tornqvist_log_index_cr4'] = (
    agg_cr4_benchmark['tornqvist_log_index_cr4']
    - float(agg_cr4_benchmark.loc[agg_cr4_benchmark['year'] == base_year_agg, 'tornqvist_log_index_cr4'].iloc[0])
)
base_cr4_level = float(
    agg_cr4_benchmark_base.loc[
        agg_cr4_benchmark_base['year'] == base_year_agg,
        'cr4_weighted_benchmark_base'
    ].iloc[0]
)
agg_cr4_benchmark['cr4_weighted_benchmark'] = base_cr4_level * np.exp(
    agg_cr4_benchmark['tornqvist_log_index_cr4']
)

agg_hhi_summary = (
    agg_hhi_dist.groupby('year', as_index=False)
    .agg(
        hhi_weighted_mean=('hhi_weighted', 'mean'),
        hhi_weighted_p10=('hhi_weighted', lambda x: np.percentile(x, 10)),
        hhi_weighted_p50=('hhi_weighted', lambda x: np.percentile(x, 50)),
        hhi_weighted_p90=('hhi_weighted', lambda x: np.percentile(x, 90)),
    )
    .merge(agg_hhi_benchmark, on='year', how='left')
)

agg_cr4_summary = (
    agg_cr4_dist.groupby('year', as_index=False)
    .agg(
        cr4_weighted_mean=('cr4_weighted', 'mean'),
        cr4_weighted_p10=('cr4_weighted', lambda x: np.percentile(x, 10)),
        cr4_weighted_p50=('cr4_weighted', lambda x: np.percentile(x, 50)),
        cr4_weighted_p90=('cr4_weighted', lambda x: np.percentile(x, 90)),
    )
    .merge(agg_cr4_benchmark[['year', 'cr4_weighted_benchmark', 'tornqvist_log_index_cr4']], on='year', how='left')
)

agg_hhi_summary



In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9), sharex='col')

# Top-left: levels, HHI
ax = axes[0, 0]
tmp = agg_hhi_summary.sort_values('year')
ax.plot(tmp['year'], tmp['hhi_weighted_mean'], marker='o', color='#4C78A8', linewidth=2.0, label='Bayesian mean')
ax.fill_between(tmp['year'], tmp['hhi_weighted_p10'], tmp['hhi_weighted_p90'], color='#4C78A8', alpha=0.2, label='Bayesian 10-90 interval')
ax.plot(tmp['year'], tmp['hhi_weighted_benchmark'], marker='v', linestyle='--', color='firebrick', linewidth=1.8, label='Benchmark point estimate')
ax.set_title('Levels: weighted aggregate HHI')
ax.set_ylabel('HHI')
ax.grid(axis='y', alpha=0.2)
ax.legend(frameon=False, loc='upper left')

# Top-right: levels, CR4
ax = axes[0, 1]
tmp = agg_cr4_summary.sort_values('year')
ax.plot(tmp['year'], tmp['cr4_weighted_mean'], marker='o', color='#59A14F', linewidth=2.0, label='Bayesian mean')
ax.fill_between(tmp['year'], tmp['cr4_weighted_p10'], tmp['cr4_weighted_p90'], color='#59A14F', alpha=0.2, label='Bayesian 10-90 interval')
ax.plot(tmp['year'], tmp['cr4_weighted_benchmark'], marker='v', linestyle='--', color='firebrick', linewidth=1.8, label='Benchmark Törnqvist chain')
ax.set_title('Levels: weighted aggregate CR4')
ax.set_ylabel('CR4')
ax.grid(axis='y', alpha=0.2)
ax.legend(frameon=False, loc='upper left')

# Bottom-left: log changes, HHI
ax = axes[1, 0]
tmp = agg_hhi_log_summary.sort_values('year')
ax.plot(tmp['year'], tmp['delta_log_hhi_mean'], marker='o', color='#4C78A8', linewidth=2.0, label='Bayesian mean')
ax.fill_between(tmp['year'], tmp['delta_log_hhi_p10'], tmp['delta_log_hhi_p90'], color='#4C78A8', alpha=0.2, label='Bayesian 10-90 interval')
ax.plot(tmp['year'], tmp['delta_log_hhi_weighted_benchmark'], marker='v', linestyle='--', color='firebrick', linewidth=1.8, label='Benchmark point estimate')
ax.set_title(f'Log change relative to {base_year_agg}: HHI')
ax.set_xlabel('Year')
ax.set_ylabel(r'log(HHI$_t$) - log(HHI$_{%d}$)' % base_year_agg)
ax.grid(axis='y', alpha=0.2)
ax.legend(frameon=False, loc='upper left')

# Bottom-right: log changes, CR4
ax = axes[1, 1]
tmp = agg_cr4_log_summary.sort_values('year')
ax.plot(tmp['year'], tmp['delta_log_cr4_mean'], marker='o', color='#59A14F', linewidth=2.0, label='Bayesian mean')
ax.fill_between(tmp['year'], tmp['delta_log_cr4_p10'], tmp['delta_log_cr4_p90'], color='#59A14F', alpha=0.2, label='Bayesian 10-90 interval')
ax.plot(tmp['year'], tmp['delta_log_cr4_weighted_benchmark'], marker='v', linestyle='--', color='firebrick', linewidth=1.8, label='Benchmark Törnqvist chain')
ax.plot(tmp['year'], tmp['delta_log_cr4_weighted_benchmark_direct'], marker='s', linestyle=':', color='#B07AA1', linewidth=1.8, label='Benchmark fixed-weight log change')
ax.set_title(f'Log change relative to {base_year_agg}: CR4')
ax.set_xlabel('Year')
ax.set_ylabel(r'log(CR4$_t$) - log(CR4$_{%d}$)' % base_year_agg)
ax.grid(axis='y', alpha=0.2)
ax.legend(frameon=False, loc='upper left')

fig.suptitle('Aggregate concentration trends: levels and log changes', y=0.98)
plt.tight_layout()
plt.show()



## Specification D. Aggregate year-weighted concentration comparison

This section builds a second aggregate specification using **time-varying annual industry sales weights**.

**Weight construction.** For each year $t$, industry weights are
$$
 w_{kt}^{\text{year}} = \frac{\text{industry sales}_{kt}}{\sum_j \text{industry sales}_{jt}}.
$$
Aggregate annual concentration is then
$$
C_t^{\text{agg, year}} = \sum_k w_{kt}^{\text{year}} C_{kt}.
$$

**Interpretation.** Unlike the fixed-weight series, this specification allows the aggregate measure to reflect both changes in within-industry concentration and changes in the relative economic importance of industries over time.

**Figure note.** The annual comparison plot should be read as a weighted macro concentration summary, not as a pure within-industry trend. If the year-weighted aggregate diverges from the fixed-weight aggregate, part of the difference is composition: industries with different concentration levels are gaining or losing sales weight over time.



In [ ]:
# Aggregate comparison using year-specific industry sales weights.
# This gives a direct annual benchmark-vs-Bayes picture with weights that change by year.
year_industry_weights = (
    benchmark_firm_industry_trend.groupby(['year', 'naics_code'], as_index=False)
    .agg(industry_sales=('industry_total_sales', 'first'))
)
year_industry_weights['industry_weight_year'] = (
    year_industry_weights['industry_sales']
    / year_industry_weights.groupby('year')['industry_sales'].transform('sum')
)

year_industry_weights = year_industry_weights.merge(
    naics_title_map,
    on='naics_code',
    how='left'
)

year_industry_weights.head()



In [ ]:
# Bayesian aggregate distributions with year-specific weights
agg_hhi_dist_year_weighted = (
    df_hhi_dist_trend.merge(
        year_industry_weights[['year', 'naics_code', 'industry_weight_year']],
        on=['year', 'naics_code'],
        how='inner'
    )
    .assign(weighted_hhi=lambda d: d['hhi'] * d['industry_weight_year'])
    .groupby(['draw', 'year'], as_index=False)
    .agg(hhi_year_weighted=('weighted_hhi', 'sum'))
)

agg_cr4_dist_year_weighted = (
    df_cr4_dist_trend.merge(
        year_industry_weights[['year', 'naics_code', 'industry_weight_year']],
        on=['year', 'naics_code'],
        how='inner'
    )
    .assign(weighted_cr4=lambda d: d['cr4'] * d['industry_weight_year'])
    .groupby(['draw', 'year'], as_index=False)
    .agg(cr4_year_weighted=('weighted_cr4', 'sum'))
)

# Benchmark annual point estimates with the same year-specific weights
agg_hhi_benchmark_year_weighted = (
    benchmark_hhi_trend.merge(
        year_industry_weights[['year', 'naics_code', 'industry_weight_year']],
        on=['year', 'naics_code'],
        how='inner'
    )
    .assign(weighted_hhi=lambda d: d['hhi_benchmark'] * d['industry_weight_year'])
    .groupby('year', as_index=False)
    .agg(hhi_year_weighted_benchmark=('weighted_hhi', 'sum'))
)

agg_cr4_benchmark_year_weighted = (
    benchmark_cr4_trend.merge(
        year_industry_weights[['year', 'naics_code', 'industry_weight_year']],
        on=['year', 'naics_code'],
        how='inner'
    )
    .assign(weighted_cr4=lambda d: d['cr4_benchmark'] * d['industry_weight_year'])
    .groupby('year', as_index=False)
    .agg(cr4_year_weighted_benchmark=('weighted_cr4', 'sum'))
)

agg_hhi_year_weighted_summary = (
    agg_hhi_dist_year_weighted.groupby('year', as_index=False)
    .agg(
        hhi_year_weighted_mean=('hhi_year_weighted', 'mean'),
        hhi_year_weighted_p10=('hhi_year_weighted', lambda x: np.percentile(x, 10)),
        hhi_year_weighted_p50=('hhi_year_weighted', lambda x: np.percentile(x, 50)),
        hhi_year_weighted_p90=('hhi_year_weighted', lambda x: np.percentile(x, 90)),
    )
    .merge(agg_hhi_benchmark_year_weighted, on='year', how='left')
)

agg_cr4_year_weighted_summary = (
    agg_cr4_dist_year_weighted.groupby('year', as_index=False)
    .agg(
        cr4_year_weighted_mean=('cr4_year_weighted', 'mean'),
        cr4_year_weighted_p10=('cr4_year_weighted', lambda x: np.percentile(x, 10)),
        cr4_year_weighted_p50=('cr4_year_weighted', lambda x: np.percentile(x, 50)),
        cr4_year_weighted_p90=('cr4_year_weighted', lambda x: np.percentile(x, 90)),
    )
    .merge(agg_cr4_benchmark_year_weighted, on='year', how='left')
)

agg_hhi_year_weighted_summary



In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5.4), sharex=True)

tmp = agg_hhi_year_weighted_summary.sort_values('year')
axes[0].plot(tmp['year'], tmp['hhi_year_weighted_mean'], marker='o', color='#4C78A8', linewidth=2.0, label='Bayesian mean')
#axes[0].fill_between(tmp['year'], tmp['hhi_year_weighted_p10'], tmp['hhi_year_weighted_p90'], color='#4C78A8', alpha=0.22, label='Bayesian 10-90 interval')
axes[0].plot(tmp['year'], tmp['hhi_year_weighted_benchmark'], marker='v', linestyle='--', color='firebrick', linewidth=1.8, label='Benchmark point estimate')
#axes[0].set_title('Year-weighted aggregate HHI by year')
axes[0].set_xlabel('Year')
axes[0].set_ylabel('HHI')
axes[0].grid(axis='y', alpha=0.2)
axes[0].legend(frameon=False, loc='upper left')

tmp = agg_cr4_year_weighted_summary.sort_values('year')
axes[1].plot(tmp['year'], tmp['cr4_year_weighted_mean'], marker='o', color='#59A14F', linewidth=2.0, label='Bayesian mean')
#axes[1].fill_between(tmp['year'], tmp['cr4_year_weighted_p10'], tmp['cr4_year_weighted_p90'], color='#59A14F', alpha=0.22, label='Bayesian 10-90 interval')
axes[1].plot(tmp['year'], tmp['cr4_year_weighted_benchmark'], marker='v', linestyle='--', color='firebrick', linewidth=1.8, label='Benchmark point estimate')
#axes[1].set_title('Year-weighted aggregate CR4 by year')
axes[1].set_xlabel('Year')
axes[1].set_ylabel('CR4')
axes[1].grid(axis='y', alpha=0.2)
axes[1].legend(frameon=False, loc='upper left')

plt.tight_layout()
plt.savefig(f'{figure_output}/aggregate_concentration_trends_level_year_weighted.png', dpi=300)
plt.show()



In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5.4), sharex=True)

# Log growth for year-weighted aggregate HHI, relative to the first available year.
tmp = agg_hhi_year_weighted_summary.sort_values('year').copy()
base_hhi_mean = tmp['hhi_year_weighted_mean'].iloc[0]
base_hhi_p10 = tmp['hhi_year_weighted_p10'].iloc[0]
base_hhi_p90 = tmp['hhi_year_weighted_p90'].iloc[0]
base_hhi_benchmark = tmp['hhi_year_weighted_benchmark'].iloc[0]

tmp['delta_log_hhi_year_weighted_mean'] = np.log(tmp['hhi_year_weighted_mean']) - np.log(base_hhi_mean)
tmp['delta_log_hhi_year_weighted_p10'] = np.log(tmp['hhi_year_weighted_p10']) - np.log(base_hhi_p10)
tmp['delta_log_hhi_year_weighted_p90'] = np.log(tmp['hhi_year_weighted_p90']) - np.log(base_hhi_p90)
tmp['delta_log_hhi_year_weighted_benchmark'] = np.log(tmp['hhi_year_weighted_benchmark']) - np.log(base_hhi_benchmark)

axes[0].plot(tmp['year'], tmp['delta_log_hhi_year_weighted_mean'], marker='o', color='#4C78A8', linewidth=2.0, label='Bayesian mean')
#axes[0].fill_between(tmp['year'], tmp['delta_log_hhi_year_weighted_p10'], tmp['delta_log_hhi_year_weighted_p90'], color='#4C78A8', alpha=0.22, label='Bayesian 10-90 interval')
axes[0].plot(tmp['year'], tmp['delta_log_hhi_year_weighted_benchmark'], marker='v', linestyle='--', color='firebrick', linewidth=1.8, label='Benchmark point estimate')
#axes[0].set_title(f'Year-weighted aggregate HHI log growth relative to {int(tmp["year"].iloc[0])}')
axes[0].set_xlabel('Year')
axes[0].set_ylabel(r'log(HHI$_t$) - log(HHI$_{%d}$)' % int(tmp['year'].iloc[0]))
axes[0].grid(axis='y', alpha=0.2)
axes[0].legend(frameon=False, loc='upper left')

# Log growth for year-weighted aggregate CR4, relative to the first available year.
tmp = agg_cr4_year_weighted_summary.sort_values('year').copy()
base_cr4_mean = tmp['cr4_year_weighted_mean'].iloc[0]
base_cr4_p10 = tmp['cr4_year_weighted_p10'].iloc[0]
base_cr4_p90 = tmp['cr4_year_weighted_p90'].iloc[0]
base_cr4_benchmark = tmp['cr4_year_weighted_benchmark'].iloc[0]

tmp['delta_log_cr4_year_weighted_mean'] = np.log(tmp['cr4_year_weighted_mean']) - np.log(base_cr4_mean)
tmp['delta_log_cr4_year_weighted_p10'] = np.log(tmp['cr4_year_weighted_p10']) - np.log(base_cr4_p10)
tmp['delta_log_cr4_year_weighted_p90'] = np.log(tmp['cr4_year_weighted_p90']) - np.log(base_cr4_p90)
tmp['delta_log_cr4_year_weighted_benchmark'] = np.log(tmp['cr4_year_weighted_benchmark']) - np.log(base_cr4_benchmark)

axes[1].plot(tmp['year'], tmp['delta_log_cr4_year_weighted_mean'], marker='o', color='#59A14F', linewidth=2.0, label='Bayesian mean')
#[1].fill_between(tmp['year'], tmp['delta_log_cr4_year_weighted_p10'], tmp['delta_log_cr4_year_weighted_p90'], color='#59A14F', alpha=0.22, label='Bayesian 10-90 interval')
axes[1].plot(tmp['year'], tmp['delta_log_cr4_year_weighted_benchmark'], marker='v', linestyle='--', color='firebrick', linewidth=1.8, label='Benchmark point estimate')
#axes[1].set_title(f'Year-weighted aggregate CR4 log growth relative to {int(tmp["year"].iloc[0])}')
axes[1].set_xlabel('Year')
axes[1].set_ylabel(r'log(CR4$_t$) - log(CR4$_{%d}$)' % int(tmp['year'].iloc[0]))
axes[1].grid(axis='y', alpha=0.2)
axes[1].legend(frameon=False, loc='upper left')

plt.tight_layout()
plt.savefig(f'{figure_output}/aggregate_concentration_trends_log_year_weighted.png', dpi=300)
plt.show()



## Figure note. 2025 aggregate weighted posterior versus benchmark

These figures collapse the entire cross-section into a **single 2025 weighted concentration object** for HHI and CR4.

**Object plotted.** The posterior density is the distribution of the year-weighted aggregate concentration statistic across Dirichlet draws. The triangle is the deterministic benchmark point estimate using the same 2025 industry sales weights.

**Interpretation.** If the benchmark point estimate lies outside the posterior bulk, the result should be read as evidence that deterministic firm-industry assignment is materially inconsistent with the uncertainty-aware allocation model at the aggregate level. If it lies inside the posterior bulk, the deterministic benchmark is broadly compatible with the Bayesian allocation uncertainty for that year.



In [ ]:
aggregate_target_year = 2025

agg_hhi_dist_2025 = agg_hhi_dist_year_weighted[
    agg_hhi_dist_year_weighted['year'] == aggregate_target_year
].copy()
agg_cr4_dist_2025 = agg_cr4_dist_year_weighted[
    agg_cr4_dist_year_weighted['year'] == aggregate_target_year
].copy()

agg_hhi_benchmark_2025 = float(
    agg_hhi_benchmark_year_weighted.loc[
        agg_hhi_benchmark_year_weighted['year'] == aggregate_target_year,
        'hhi_year_weighted_benchmark'
    ].iloc[0]
)
agg_cr4_benchmark_2025 = float(
    agg_cr4_benchmark_year_weighted.loc[
        agg_cr4_benchmark_year_weighted['year'] == aggregate_target_year,
        'cr4_year_weighted_benchmark'
    ].iloc[0]
)

agg_hhi_dist_2025.describe()



In [ ]:
plt.figure(figsize=(8.6, 5.2))
ax = plt.gca()

sns.kdeplot(
    data=agg_hhi_dist_2025,
    x='hhi_year_weighted',
    fill=True,
    color='#4C78A8',
    alpha=0.28,
    linewidth=0,
    ax=ax
)
sns.kdeplot(
    data=agg_hhi_dist_2025,
    x='hhi_year_weighted',
    color='#4C78A8',
    linewidth=2.2,
    ax=ax,
    label='Bayesian posterior'
)

hhi_median_2025 = float(agg_hhi_dist_2025['hhi_year_weighted'].median())
ax.axvline(hhi_median_2025, color='#4C78A8', linestyle='--', linewidth=2, label='Posterior median')
ax.scatter(
    [agg_hhi_benchmark_2025],
    [0],
    marker='v',
    s=150,
    color='firebrick',
    zorder=5,
    label='Benchmark point estimate'
)

#ax.set_title('2025 year-weighted aggregate HHI')
ax.set_xlabel('Weighted HHI (2025)')
ax.set_ylabel('Posterior density')
ax.grid(axis='y', alpha=0.2)
ax.legend(frameon=False, loc='upper left')
plt.tight_layout()
plt.savefig(f'{figure_output}/hhi_distribution_2025.png', dpi=300)
plt.show()



In [ ]:
plt.figure(figsize=(8.6, 5.2))
ax = plt.gca()

sns.kdeplot(
    data=agg_cr4_dist_2025,
    x='cr4_year_weighted',
    fill=True,
    color='#59A14F',
    alpha=0.28,
    linewidth=0,
    ax=ax
)
sns.kdeplot(
    data=agg_cr4_dist_2025,
    x='cr4_year_weighted',
    color='#59A14F',
    linewidth=2.2,
    ax=ax,
    label='Bayesian posterior'
)

cr4_median_2025 = float(agg_cr4_dist_2025['cr4_year_weighted'].median())
ax.axvline(cr4_median_2025, color='#59A14F', linestyle='--', linewidth=2, label='Posterior median')
ax.scatter(
    [agg_cr4_benchmark_2025],
    [0],
    marker='v',
    s=150,
    color='firebrick',
    zorder=5,
    label='Benchmark point estimate'
)

#ax.set_title('2025 year-weighted aggregate CR4')
ax.set_xlabel('Weighted CR4 (2025)')
ax.set_ylabel('Posterior density')
ax.grid(axis='y', alpha=0.2)
ax.legend(frameon=False, loc='upper left')
plt.tight_layout()
plt.savefig(f'{figure_output}/cr4_distribution_2025.png', dpi=300)
plt.show()



## Figure note. Segment-level Dirichlet posterior visualization

The purpose of this block is to show the object that sits underneath the concentration distributions: a **segment-level posterior allocation vector**.

**Selection rule.** The notebook automatically selects a large 2025 segment with at least three industry links so that the posterior is economically meaningful and visually non-degenerate.

**What is shown.** The left panel reports posterior mean industry shares for the selected segment. The right panel plots the marginal posterior densities of those shares, which summarize both central tendency and uncertainty in the segment's industry allocation.

**Interpretation.** Wide or overlapping densities indicate genuine ambiguity in how segment sales should be mapped across industries. This is the uncertainty that the deterministic benchmark suppresses and the Bayesian CR4/HHI measures propagate upward into concentration estimates.



In [ ]:
# Visualize one segment-level Dirichlet posterior underlying the Bayesian allocation model.
dirichlet_year = 2025

# Rebuild the posterior mass table for the selected year so we can inspect one segment directly.
df_year_dirichlet = df_sales_count_merged[df_sales_count_merged['year'] == dirichlet_year].copy()
df_evidence_dirichlet = (
    df_year_dirichlet.groupby(
        ['gvkey', 'conm', 'firm_segment_name', 'naics_code', 'sales'],
        as_index=False
    )
    .agg(evidence_mass=('sales_weighted_count', 'sum'))
)

df_post_dirichlet = df_evidence_dirichlet.merge(
    df_prior,
    on=['gvkey', 'conm', 'firm_segment_name', 'naics_code'],
    how='outer'
)

df_post_dirichlet['conm'] = (
    df_post_dirichlet['conm_x'].combine_first(df_post_dirichlet['conm_y'])
    if 'conm_x' in df_post_dirichlet.columns else df_post_dirichlet['conm']
)
if 'conm_x' in df_post_dirichlet.columns:
    df_post_dirichlet = df_post_dirichlet.drop(columns=['conm_x', 'conm_y'])

df_post_dirichlet['sales'] = df_post_dirichlet['sales'].fillna(0)
df_post_dirichlet['evidence_mass'] = df_post_dirichlet['evidence_mass'].fillna(0)
df_post_dirichlet['prior_mass'] = df_post_dirichlet['prior_mass'].fillna(SMOOTH)
df_post_dirichlet['alpha_post'] = df_post_dirichlet['prior_mass'] + df_post_dirichlet['evidence_mass']

# Pick an economically important multi-industry segment automatically.
dirichlet_focus_candidates = (
    df_post_dirichlet.groupby(['gvkey', 'conm', 'firm_segment_name'], as_index=False)
    .agg(
        sales=('sales', 'max'),
        n_industries=('naics_code', 'nunique'),
        posterior_mass=('alpha_post', 'sum')
    )
)
dirichlet_focus_candidates = dirichlet_focus_candidates[
    (dirichlet_focus_candidates['sales'] > 0) & (dirichlet_focus_candidates['n_industries'] >= 3)
].copy()
dirichlet_focus = dirichlet_focus_candidates.sort_values('sales', ascending=False).iloc[0]

dirichlet_focus_segment = df_post_dirichlet[
    (df_post_dirichlet['gvkey'] == dirichlet_focus['gvkey'])
    & (df_post_dirichlet['conm'] == dirichlet_focus['conm'])
    & (df_post_dirichlet['firm_segment_name'] == dirichlet_focus['firm_segment_name'])
].copy().sort_values('alpha_post', ascending=False)

dirichlet_focus_segment[['conm', 'firm_segment_name', 'naics_code', 'sales', 'prior_mass', 'evidence_mass', 'alpha_post']]



In [ ]:
dirichlet_n_draws = 5000
alpha_focus = dirichlet_focus_segment['alpha_post'].to_numpy(dtype=float)
industries_focus = dirichlet_focus_segment['naics_code'].astype(str).tolist()

theta_focus_draws = np.random.dirichlet(alpha_focus, size=dirichlet_n_draws)

df_dirichlet_draws = pd.DataFrame(theta_focus_draws, columns=industries_focus)
posterior_mean_focus = df_dirichlet_draws.mean().sort_values(ascending=False)

top_k = min(5, len(posterior_mean_focus))
top_industries_focus = posterior_mean_focus.head(top_k).index.tolist()

if len(posterior_mean_focus) > top_k:
    df_dirichlet_plot = df_dirichlet_draws[top_industries_focus].copy()
    df_dirichlet_plot['Other'] = 1 - df_dirichlet_plot.sum(axis=1)
    plot_order_focus = top_industries_focus + ['Other']
else:
    df_dirichlet_plot = df_dirichlet_draws[top_industries_focus].copy()
    plot_order_focus = top_industries_focus.copy()

posterior_mean_plot = df_dirichlet_plot[plot_order_focus].mean().sort_values(ascending=False)
df_dirichlet_long = df_dirichlet_plot[plot_order_focus].melt(var_name='naics_code', value_name='posterior_share')

posterior_mean_plot



In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5.4))

# Left: posterior mean shares
bar_df = posterior_mean_plot.reset_index()
bar_df.columns = ['naics_code', 'posterior_mean_share']
sns.barplot(data=bar_df, x='posterior_mean_share', y='naics_code', color='#4C78A8', ax=axes[0])
axes[0].set_title(
    f"\n{dirichlet_focus['firm_segment_name']} ({dirichlet_year})"
)
axes[0].set_xlabel('Posterior mean share')
axes[0].set_ylabel('NAICS code')
axes[0].grid(axis='x', alpha=0.2)

# Right: marginal posterior densities for the main industries
palette = sns.color_palette('deep', n_colors=len(plot_order_focus))
for color, naics_code in zip(palette, plot_order_focus):
    sns.kdeplot(
        data=df_dirichlet_long[df_dirichlet_long['naics_code'] == naics_code],
        x='posterior_share',
        fill=True,
        alpha=0.15,
        linewidth=0,
        color=color,
        ax=axes[1]
    )
    sns.kdeplot(
        data=df_dirichlet_long[df_dirichlet_long['naics_code'] == naics_code],
        x='posterior_share',
        linewidth=2,
        color=color,
        label=naics_code,
        ax=axes[1]
    )

#axes[1].set_title('Dirichlet marginal posterior densities')
axes[1].set_xlabel('Posterior share')
axes[1].set_ylabel('Density')
axes[1].grid(axis='y', alpha=0.2)
axes[1].legend(frameon=False, loc='upper left', title='NAICS')

plt.tight_layout()
plt.savefig(f'{figure_output}/dirichlet_posterior_example_{dirichlet_year}.png', dpi=300)
plt.show()



## Specification E. Aggregate log-change comparison

This section studies concentration growth relative to the first available year in the aggregate series.

**Transformation.** For any concentration measure $C_t$, the plotted object is
$$
\Delta \log C_t = \log(C_t) - \log(C_{t_0}),
$$
where $t_0$ is the first available post-filter year in the aggregate series.

**Benchmark CR4 note.** For CR4, the benchmark log-change series is constructed from the Törnqvist chain index computed above rather than from a direct log-difference of a fixed-weight average. This is deliberate: it preserves a chain-linked growth interpretation when industry sales shares vary over time.

**Figure note.** A benchmark line systematically below the Bayesian posterior band means the deterministic benchmark implies weaker concentration growth than the uncertainty-aware allocation model. A benchmark line above the band would imply the reverse.



In [ ]:
base_hhi = agg_hhi_dist[agg_hhi_dist['year'] == base_year_agg][['draw', 'hhi_weighted']].rename(columns={'hhi_weighted': 'hhi_base'})
agg_hhi_log = agg_hhi_dist.merge(base_hhi, on='draw', how='inner')
agg_hhi_log['delta_log_hhi_weighted'] = np.log(agg_hhi_log['hhi_weighted']) - np.log(agg_hhi_log['hhi_base'])

base_cr4 = agg_cr4_dist[agg_cr4_dist['year'] == base_year_agg][['draw', 'cr4_weighted']].rename(columns={'cr4_weighted': 'cr4_base'})
agg_cr4_log = agg_cr4_dist.merge(base_cr4, on='draw', how='inner')
agg_cr4_log['delta_log_cr4_weighted'] = np.log(agg_cr4_log['cr4_weighted']) - np.log(agg_cr4_log['cr4_base'])

base_hhi_benchmark = float(agg_hhi_benchmark.loc[agg_hhi_benchmark['year'] == base_year_agg, 'hhi_weighted_benchmark'].iloc[0])
agg_hhi_benchmark_log = agg_hhi_benchmark.copy()
agg_hhi_benchmark_log['delta_log_hhi_weighted_benchmark'] = np.log(agg_hhi_benchmark_log['hhi_weighted_benchmark']) - np.log(base_hhi_benchmark)

agg_cr4_benchmark_log = agg_cr4_benchmark[['year', 'tornqvist_log_index_cr4']].rename(
    columns={'tornqvist_log_index_cr4': 'delta_log_cr4_weighted_benchmark'}
)

# Direct fixed-weight CR4 benchmark log change, without Törnqvist chaining.
base_cr4_benchmark_direct = float(
    agg_cr4_benchmark_base.loc[
        agg_cr4_benchmark_base['year'] == base_year_agg,
        'cr4_weighted_benchmark_base'
    ].iloc[0]
)
agg_cr4_benchmark_direct_log = agg_cr4_benchmark_base.copy()
agg_cr4_benchmark_direct_log['delta_log_cr4_weighted_benchmark_direct'] = (
    np.log(agg_cr4_benchmark_direct_log['cr4_weighted_benchmark_base'])
    - np.log(base_cr4_benchmark_direct)
)

agg_hhi_log_summary = (
    agg_hhi_log.groupby('year', as_index=False)
    .agg(
        delta_log_hhi_mean=('delta_log_hhi_weighted', 'mean'),
        delta_log_hhi_p10=('delta_log_hhi_weighted', lambda x: np.percentile(x, 10)),
        delta_log_hhi_p50=('delta_log_hhi_weighted', lambda x: np.percentile(x, 50)),
        delta_log_hhi_p90=('delta_log_hhi_weighted', lambda x: np.percentile(x, 90)),
    )
    .merge(agg_hhi_benchmark_log[['year', 'delta_log_hhi_weighted_benchmark']], on='year', how='left')
)

agg_cr4_log_summary = (
    agg_cr4_log.groupby('year', as_index=False)
    .agg(
        delta_log_cr4_mean=('delta_log_cr4_weighted', 'mean'),
        delta_log_cr4_p10=('delta_log_cr4_weighted', lambda x: np.percentile(x, 10)),
        delta_log_cr4_p50=('delta_log_cr4_weighted', lambda x: np.percentile(x, 50)),
        delta_log_cr4_p90=('delta_log_cr4_weighted', lambda x: np.percentile(x, 90)),
    )
    .merge(agg_cr4_benchmark_log, on='year', how='left')
    .merge(
        agg_cr4_benchmark_direct_log[['year', 'delta_log_cr4_weighted_benchmark_direct']],
        on='year',
        how='left'
    )
)

agg_hhi_log_summary



In [ ]:
print('Log-change panels are now included in the 2x2 aggregate trend figure above (levels on top; log changes on bottom; HHI left; CR4 right).')



In [ ]:
target_year = 2025

compare_hhi_2025 = compare_hhi[compare_hhi['year'] == target_year].copy()
compare_cr4_2025 = compare_cr4[compare_cr4['year'] == target_year].copy()

# Keep only industries where the benchmark point estimate falls inside the Bayesian 10-90 range.
credible_hhi_2025 = compare_hhi_2025[
    (compare_hhi_2025['hhi_benchmark'] >= compare_hhi_2025['hhi_p10'])
    & (compare_hhi_2025['hhi_benchmark'] <= compare_hhi_2025['hhi_p90'])
].copy()

credible_cr4_2025 = compare_cr4_2025[
    (compare_cr4_2025['cr4_benchmark'] >= compare_cr4_2025['cr4_p10'])
    & (compare_cr4_2025['cr4_benchmark'] <= compare_cr4_2025['cr4_p90'])
].copy()

# Pick a small set of credible industries to visualize, prioritizing economically large ones.
benchmark_sales_2025 = (
    benchmark_firm_industry[benchmark_firm_industry['year'] == target_year]
    .groupby('naics_code', as_index=False)
    .agg(industry_total_sales=('industry_total_sales', 'mean'))
)

credible_hhi_2025 = credible_hhi_2025.merge(benchmark_sales_2025, on='naics_code', how='left')
selected_naics_codes = (
    credible_hhi_2025.sort_values('industry_total_sales', ascending=False)
    .head(4)['naics_code']
    .tolist()
)

credible_hhi_2025[credible_hhi_2025['naics_code'].isin(selected_naics_codes)][['naics_code', 'naics_title', 'hhi_benchmark', 'hhi_mean', 'hhi_p10', 'hhi_p90', 'industry_total_sales']].sort_values('industry_total_sales', ascending=False)

## Figure note. All-industry 2025 level comparison

This section compares benchmark and Bayesian concentration measures across the full 2025 industry cross-section.

**Units.** Each observation is a 3-digit NAICS industry. The interval plots show the Bayesian 10--90 posterior range and mean, while the scatterplots compare the Bayesian posterior mean directly to the deterministic benchmark point estimate.

**Why two displays?** The interval plot emphasizes uncertainty and ranking across industries. The scatterplot emphasizes systematic bias: points below the 45-degree line indicate industries where the Bayesian mean is below the benchmark; points above indicate the opposite.

**Interpretation caution.** Because the selected plotting subset is ordered by industry sales, these figures are best read as a comparison among the economically largest observed industries rather than as a census of every NAICS cell with equal substantive importance.



In [ ]:
# All-industry level comparison for the target year.
all_hhi_2025 = compare_hhi_2025.dropna(subset=['hhi_benchmark', 'hhi_mean', 'hhi_p10', 'hhi_p90']).copy()
all_cr4_2025 = compare_cr4_2025.dropna(subset=['cr4_benchmark', 'cr4_mean', 'cr4_p10', 'cr4_p90']).copy()

all_hhi_2025 = all_hhi_2025.merge(benchmark_sales_2025, on='naics_code', how='left')
all_cr4_2025 = all_cr4_2025.merge(benchmark_sales_2025, on='naics_code', how='left')

all_hhi_2025['label'] = all_hhi_2025['naics_code'].astype(str) + ' - ' + all_hhi_2025['naics_title'].fillna('')
all_cr4_2025['label'] = all_cr4_2025['naics_code'].astype(str) + ' - ' + all_cr4_2025['naics_title'].fillna('')

# Set to None to show every industry.
max_industries_to_plot = 25
if max_industries_to_plot is not None:
    all_hhi_plot = all_hhi_2025.sort_values('industry_total_sales', ascending=False).head(max_industries_to_plot).copy()
    all_cr4_plot = all_cr4_2025.sort_values('industry_total_sales', ascending=False).head(max_industries_to_plot).copy()
else:
    all_hhi_plot = all_hhi_2025.copy()
    all_cr4_plot = all_cr4_2025.copy()

all_hhi_plot[['naics_code', 'naics_title', 'hhi_benchmark', 'hhi_mean', 'hhi_p10', 'hhi_p90', 'industry_total_sales']].head()


In [ ]:
plot_df = all_hhi_plot.sort_values('hhi_mean', ascending=True).reset_index(drop=True)
y = np.arange(len(plot_df))

plt.figure(figsize=(12, max(8, 0.35 * len(plot_df))))
plt.hlines(y=y, xmin=plot_df['hhi_p10'], xmax=plot_df['hhi_p90'], color='#4C78A8', linewidth=2.5, alpha=0.85, label='Bayesian 10-90 interval')
plt.scatter(plot_df['hhi_mean'], y, color='#4C78A8', s=35, zorder=3, label='Bayesian mean')
plt.scatter(plot_df['hhi_benchmark'], y, color='firebrick', marker='v', s=80, zorder=4, label='Benchmark point estimate')

plt.yticks(y, plot_df['label'])
plt.xlabel(f'HHI ({target_year})')
#plt.title(f'Level comparison: Bayesian HHI vs benchmark across industries ({target_year})')
plt.legend(frameon=False, loc='best')
plt.tight_layout()
plt.savefig(f'{figure_output}/hhi_comparison_by_industry_2025.png', dpi=300)
plt.show()


In [ ]:
plot_df = all_cr4_plot.sort_values('cr4_mean', ascending=True).reset_index(drop=True)
y = np.arange(len(plot_df))

plt.figure(figsize=(12, max(8, 0.35 * len(plot_df))))
plt.hlines(y=y, xmin=plot_df['cr4_p10'], xmax=plot_df['cr4_p90'], color='#59A14F', linewidth=2.5, alpha=0.85, label='Bayesian 10-90 interval')
plt.scatter(plot_df['cr4_mean'], y, color='#59A14F', s=35, zorder=3, label='Bayesian mean')
plt.scatter(plot_df['cr4_benchmark'], y, color='firebrick', marker='v', s=80, zorder=4, label='Benchmark point estimate')

plt.yticks(y, plot_df['label'])
plt.xlabel(f'CR4 ({target_year})')
#plt.title(f'Level comparison: Bayesian CR4 vs benchmark across industries ({target_year})')
plt.legend(frameon=False, loc='best')
plt.tight_layout()
plt.savefig(f'{figure_output}/cr4_comparison_by_industry_2025.png', dpi=300)
plt.show()


In [ ]:
plot_df = all_hhi_plot.dropna(subset=['hhi_benchmark', 'hhi_mean']).copy()

plt.figure(figsize=(8, 8))
plt.scatter(plot_df['hhi_benchmark'], plot_df['hhi_mean'], alpha=0.7, color='#4C78A8', label='Industry')

line_min = min(plot_df['hhi_benchmark'].min(), plot_df['hhi_mean'].min())
line_max = max(plot_df['hhi_benchmark'].max(), plot_df['hhi_mean'].max())
plt.plot([line_min, line_max], [line_min, line_max], linestyle='--', color='gray', linewidth=1.5, label='45-degree line')

for _, row in plot_df.iterrows():
    plt.annotate(str(row['naics_code']), (row['hhi_benchmark'], row['hhi_mean']), fontsize=8, alpha=0.8)

plt.xlabel('Benchmark HHI')
plt.ylabel('Bayesian mean HHI')
#plt.title(f'All-industry level comparison: benchmark vs Bayesian mean HHI ({target_year})')
plt.legend(frameon=False, loc='upper left')
plt.tight_layout()
plt.savefig(f'{figure_output}/hhi_scatter_comparison_2025.png', dpi=300)
plt.show()




In [ ]:
plot_df = all_cr4_plot.dropna(subset=['cr4_benchmark', 'cr4_mean']).copy()

plt.figure(figsize=(8, 8))
plt.scatter(plot_df['cr4_benchmark'], plot_df['cr4_mean'], alpha=0.7, color='#59A14F', label='Industry')

line_min = min(plot_df['cr4_benchmark'].min(), plot_df['cr4_mean'].min())
line_max = max(plot_df['cr4_benchmark'].max(), plot_df['cr4_mean'].max())
plt.plot([line_min, line_max], [line_min, line_max], linestyle='--', color='gray', linewidth=1.5, label='45-degree line')

for _, row in plot_df.iterrows():
    plt.annotate(str(row['naics_code']), (row['cr4_benchmark'], row['cr4_mean']), fontsize=8, alpha=0.8)

plt.xlabel('Benchmark CR4')
plt.ylabel('Bayesian mean CR4')
#plt.title(f'All-industry level comparison: benchmark vs Bayesian mean CR4 ({target_year})')
plt.legend(frameon=False, loc='upper left')
plt.tight_layout()
plt.savefig(f'{figure_output}/cr4_scatter_comparison_2025.png', dpi=300)
plt.show()




## Figure note. Selected-industry 2025 posterior density comparisons

These figures are the paper-style visual comparison between the Bayesian posterior distribution and the deterministic benchmark point estimate for a small number of economically important industries.

**Selection rule.** Industries are filtered to those where the benchmark point estimate lies inside the Bayesian 10--90 interval, then prioritized by 2025 industry sales. This makes the final set both economically important and visually credible for close inspection.

**Interpretation.** The dashed vertical line is the posterior median; the triangle is the benchmark point estimate. CR4 is often more sensitive than HHI to reallocations near the top-4 threshold, so it is natural for CR4 benchmark discrepancies to be larger even when HHI discrepancies remain modest.



In [ ]:
# Plot Bayesian HHI distributions one industry at a time in the style of posterior_cr4_distributions.png.
plot_hhi_dist = df_hhi_dist[(df_hhi_dist['naics_code'].isin(selected_naics_codes)) & (df_hhi_dist['year'] == target_year)].merge(
    naics_title_map, on='naics_code', how='left'
)
plot_hhi_benchmark = benchmark_hhi[
    (benchmark_hhi['naics_code'].isin(selected_naics_codes)) &
    (benchmark_hhi['year'] == target_year)
].copy()


plot_hhi_dist['label'] = plot_hhi_dist['naics_code'].astype(str) + ' - ' + plot_hhi_dist['naics_title'].fillna('')
plot_hhi_benchmark['label'] = plot_hhi_benchmark['naics_code'].astype(str) + ' - ' + plot_hhi_benchmark['naics_title'].fillna('')

label_order = (
    credible_hhi_2025[credible_hhi_2025['naics_code'].isin(selected_naics_codes)]
    .sort_values('industry_total_sales', ascending=False)
    .assign(label=lambda d: d['naics_code'].astype(str) + ' - ' + d['naics_title'].fillna(''))['label']
    .tolist()
)

palette = sns.color_palette('deep', n_colors=len(label_order))
color_map = dict(zip(label_order, palette))

for label in label_order:
    tmp = plot_hhi_dist[plot_hhi_dist['label'] == label]
    if tmp.empty:
        continue
    plt.figure(figsize=(9, 5.5))
    sns.kdeplot(data=tmp, x='hhi', fill=True, alpha=0.20, linewidth=0, color=color_map[label])
    sns.kdeplot(data=tmp, x='hhi', fill=False, linewidth=2.2, color=color_map[label], label=label)
    median_val = tmp['hhi'].median()
    plt.axvline(median_val, linestyle='--', linewidth=1.8, color=color_map[label], alpha=0.9)
    tmp_b = plot_hhi_benchmark[plot_hhi_benchmark['label'] == label]
    if tmp_b.empty:
        plt.close()
        continue
    x = tmp_b['hhi_benchmark'].iloc[0]
    plt.scatter([x], [0], marker='v', s=180, color=color_map[label], edgecolor=color_map[label], zorder=5)
    plt.xlabel(f'HHI ({target_year})')
    plt.ylabel('Posterior density')
    plt.title(f'{label}\nDashed line = posterior median; triangle = benchmark point estimate')
    plt.legend(frameon=False, loc='upper right')
    plt.tight_layout()
    plt.show()

In [ ]:
plot_cr4_dist = df_cr4_dist[(df_cr4_dist['naics_code'].isin(selected_naics_codes)) & (df_cr4_dist['year'] == target_year)].merge(
    naics_title_map, on='naics_code', how='left'
)
plot_cr4_benchmark = benchmark_cr4[
    (benchmark_cr4['naics_code'].isin(selected_naics_codes)) &
    (benchmark_cr4['year'] == target_year)
].copy()

plot_cr4_dist['label'] = plot_cr4_dist['naics_code'].astype(str) + ' - ' + plot_cr4_dist['naics_title'].fillna('')
plot_cr4_benchmark['label'] = plot_cr4_benchmark['naics_code'].astype(str) + ' - ' + plot_cr4_benchmark['naics_title'].fillna('')

for label in label_order:
    tmp = plot_cr4_dist[plot_cr4_dist['label'] == label]
    if tmp.empty:
        continue
    plt.figure(figsize=(9, 5.5))
    sns.kdeplot(data=tmp, x='cr4', fill=True, alpha=0.20, linewidth=0, color=color_map[label])
    sns.kdeplot(data=tmp, x='cr4', fill=False, linewidth=2.2, color=color_map[label], label=label)
    median_val = tmp['cr4'].median()
    plt.axvline(median_val, linestyle='--', linewidth=1.8, color=color_map[label], alpha=0.9)
    tmp_b = plot_cr4_benchmark[plot_cr4_benchmark['label'] == label]
    if tmp_b.empty:
        plt.close()
        continue
    x = tmp_b['cr4_benchmark'].iloc[0]
    plt.scatter([x], [0], marker='v', s=180, color=color_map[label], edgecolor=color_map[label], zorder=5)
    plt.xlabel(f'CR4 ({target_year})')
    plt.ylabel('Posterior density')
    plt.title(f'{label}\nDashed line = posterior median; triangle = benchmark point estimate')
    plt.legend(frameon=False, loc='upper right')
    plt.tight_layout()
    plt.show()

## Figure note. Selected-industry 2022-to-2025 log-change comparisons

These figures shift the comparison from concentration **levels** to concentration **growth**.

**Object plotted.** For each selected industry and each posterior draw,
$$
\Delta \log C = \log(C_{2025}) - \log(C_{2022}),
$$
for both HHI and CR4. The benchmark is the analogous deterministic log change.

**Interpretation.** A benchmark point estimate to the left of the posterior mass means the deterministic benchmark understates concentration growth relative to the Bayesian model; a point to the right would mean it overstates growth. Comparing these figures to the level figures clarifies whether benchmark differences are coming from baseline levels, subsequent growth, or both.



In [ ]:
base_year = 2022
compare_year = 2025

# Bayesian draw-by-draw log changes
bayes_hhi_2021 = df_hhi_dist[df_hhi_dist['year'] == base_year][['draw', 'naics_code', 'hhi']].rename(columns={'hhi': 'hhi_2021'})
bayes_hhi_2025 = df_hhi_dist[df_hhi_dist['year'] == compare_year][['draw', 'naics_code', 'hhi']].rename(columns={'hhi': 'hhi_2025'})
df_hhi_log_change = bayes_hhi_2021.merge(bayes_hhi_2025, on=['draw', 'naics_code'], how='inner')
df_hhi_log_change['delta_log_hhi'] = np.log(df_hhi_log_change['hhi_2025']) - np.log(df_hhi_log_change['hhi_2021'])

bayes_cr4_2021 = df_cr4_dist[df_cr4_dist['year'] == base_year][['draw', 'naics_code', 'cr4']].rename(columns={'cr4': 'cr4_2021'})
bayes_cr4_2025 = df_cr4_dist[df_cr4_dist['year'] == compare_year][['draw', 'naics_code', 'cr4']].rename(columns={'cr4': 'cr4_2025'})
df_cr4_log_change = bayes_cr4_2021.merge(bayes_cr4_2025, on=['draw', 'naics_code'], how='inner')
df_cr4_log_change['delta_log_cr4'] = np.log(df_cr4_log_change['cr4_2025']) - np.log(df_cr4_log_change['cr4_2021'])

# Naive point-estimate log changes
naive_hhi_2021 = benchmark_hhi[benchmark_hhi['year'] == base_year][['naics_code', 'hhi_benchmark']].rename(columns={'hhi_benchmark': 'hhi_2021_naive'})
naive_hhi_2025 = benchmark_hhi[benchmark_hhi['year'] == compare_year][['naics_code', 'hhi_benchmark']].rename(columns={'hhi_benchmark': 'hhi_2025_naive'})
naive_hhi_change = naive_hhi_2021.merge(naive_hhi_2025, on='naics_code', how='inner')
naive_hhi_change['delta_log_hhi_naive'] = np.log(naive_hhi_change['hhi_2025_naive']) - np.log(naive_hhi_change['hhi_2021_naive'])

naive_cr4_2021 = benchmark_cr4[benchmark_cr4['year'] == base_year][['naics_code', 'cr4_benchmark']].rename(columns={'cr4_benchmark': 'cr4_2021_naive'})
naive_cr4_2025 = benchmark_cr4[benchmark_cr4['year'] == compare_year][['naics_code', 'cr4_benchmark']].rename(columns={'cr4_benchmark': 'cr4_2025_naive'})
naive_cr4_change = naive_cr4_2021.merge(naive_cr4_2025, on='naics_code', how='inner')
naive_cr4_change['delta_log_cr4_naive'] = np.log(naive_cr4_change['cr4_2025_naive']) - np.log(naive_cr4_change['cr4_2021_naive'])

df_hhi_log_change.head()

In [ ]:
change_hhi_summary = (
    df_hhi_log_change.groupby('naics_code', as_index=False)
    .agg(
        delta_log_hhi_mean=('delta_log_hhi', 'mean'),
        delta_log_hhi_p10=('delta_log_hhi', lambda x: np.percentile(x, 10)),
        delta_log_hhi_p50=('delta_log_hhi', lambda x: np.percentile(x, 50)),
        delta_log_hhi_p90=('delta_log_hhi', lambda x: np.percentile(x, 90)),
    )
    .merge(naive_hhi_change[['naics_code', 'delta_log_hhi_naive']], on='naics_code', how='left')
    .merge(naics_title_map, on='naics_code', how='left')
)

change_cr4_summary = (
    df_cr4_log_change.groupby('naics_code', as_index=False)
    .agg(
        delta_log_cr4_mean=('delta_log_cr4', 'mean'),
        delta_log_cr4_p10=('delta_log_cr4', lambda x: np.percentile(x, 10)),
        delta_log_cr4_p50=('delta_log_cr4', lambda x: np.percentile(x, 50)),
        delta_log_cr4_p90=('delta_log_cr4', lambda x: np.percentile(x, 90)),
    )
    .merge(naive_cr4_change[['naics_code', 'delta_log_cr4_naive']], on='naics_code', how='left')
    .merge(naics_title_map, on='naics_code', how='left')
)

change_hhi_summary[change_hhi_summary['naics_code'].isin(selected_naics_codes)][['naics_code', 'naics_title', 'delta_log_hhi_naive', 'delta_log_hhi_mean', 'delta_log_hhi_p10', 'delta_log_hhi_p90']]

In [ ]:
plot_hhi_change = df_hhi_log_change[df_hhi_log_change['naics_code'].isin(selected_naics_codes)].merge(
    naics_title_map, on='naics_code', how='left'
)
plot_hhi_change['label'] = plot_hhi_change['naics_code'].astype(str) + ' - ' + plot_hhi_change['naics_title'].fillna('')

plot_hhi_change_naive = naive_hhi_change[naive_hhi_change['naics_code'].isin(selected_naics_codes)].merge(
    naics_title_map, on='naics_code', how='left'
)
plot_hhi_change_naive['label'] = plot_hhi_change_naive['naics_code'].astype(str) + ' - ' + plot_hhi_change_naive['naics_title'].fillna('')

for label in label_order:
    tmp = plot_hhi_change[plot_hhi_change['label'] == label]
    if tmp.empty:
        continue
    plt.figure(figsize=(9, 5.5))
    sns.kdeplot(data=tmp, x='delta_log_hhi', fill=True, alpha=0.20, linewidth=0, color=color_map[label])
    sns.kdeplot(data=tmp, x='delta_log_hhi', fill=False, linewidth=2.2, color=color_map[label], label=label)
    median_val = tmp['delta_log_hhi'].median()
    plt.axvline(median_val, linestyle='--', linewidth=1.8, color=color_map[label], alpha=0.9)
    tmp_b = plot_hhi_change_naive[plot_hhi_change_naive['label'] == label]
    if tmp_b.empty:
        plt.close()
        continue
    x = tmp_b['delta_log_hhi_naive'].iloc[0]
    plt.scatter([x], [0], marker='v', s=180, color=color_map[label], edgecolor=color_map[label], zorder=5)
    plt.xlabel(f'log(HHI {compare_year}) - log(HHI {base_year})')
    plt.ylabel('Posterior density')
    plt.title(f'{label}\nDashed line = posterior median; triangle = benchmark point estimate')
    plt.legend(frameon=False, loc='upper right')
    plt.tight_layout()
    plt.show()

In [ ]:
plot_cr4_change = df_cr4_log_change[df_cr4_log_change['naics_code'].isin(selected_naics_codes)].merge(
    naics_title_map, on='naics_code', how='left'
)
plot_cr4_change['label'] = plot_cr4_change['naics_code'].astype(str) + ' - ' + plot_cr4_change['naics_title'].fillna('')

plot_cr4_change_naive = naive_cr4_change[naive_cr4_change['naics_code'].isin(selected_naics_codes)].merge(
    naics_title_map, on='naics_code', how='left'
)
plot_cr4_change_naive['label'] = plot_cr4_change_naive['naics_code'].astype(str) + ' - ' + plot_cr4_change_naive['naics_title'].fillna('')

for label in label_order:
    tmp = plot_cr4_change[plot_cr4_change['label'] == label]
    if tmp.empty:
        continue
    plt.figure(figsize=(9, 5.5))
    sns.kdeplot(data=tmp, x='delta_log_cr4', fill=True, alpha=0.20, linewidth=0, color=color_map[label])
    sns.kdeplot(data=tmp, x='delta_log_cr4', fill=False, linewidth=2.2, color=color_map[label], label=label)
    median_val = tmp['delta_log_cr4'].median()
    plt.axvline(median_val, linestyle='--', linewidth=1.8, color=color_map[label], alpha=0.9)
    tmp_b = plot_cr4_change_naive[plot_cr4_change_naive['label'] == label]
    if tmp_b.empty:
        plt.close()
        continue
    x = tmp_b['delta_log_cr4_naive'].iloc[0]
    plt.scatter([x], [0], marker='v', s=180, color=color_map[label], edgecolor=color_map[label], zorder=5)
    plt.xlabel(f'log(CR4 {compare_year}) - log(CR4 {base_year})')
    plt.ylabel('Posterior density')
    plt.title(f'{label}\nDashed line = posterior median; triangle = benchmark point estimate')
    plt.legend(frameon=False, loc='upper right')
    plt.tight_layout()
    plt.show()

## Table export note. Publication-style WRDS summary tables

The next cell exports economics-style LaTeX summary tables directly from this notebook into `output/tables/wrds/`. The outputs match the processed WRDS files used in the analysis notebook and separate sales statistics, count statistics, and yearly panel coverage.

Generated files:
- `output/tables/wrds/wrds_sales_summary.tex`
- `output/tables/wrds/wrds_count_summary.tex`
- `output/tables/wrds/wrds_coverage_by_year.tex`
- `output/tables/wrds/wrds_summary_tables.tex`


In [ ]:
repo_root

In [ ]:
script_path = os.path.join(repo_root, 'src', 'wrds', 'generate_summary_tables.py')
runpy.run_path(str(script_path), run_name='__main__')

table_dir = os.path.join(repo_root, 'output', 'tables', 'wrds')
generated_tables = sorted(p.name for p in Path(table_dir).glob('wrds*.tex'))
print('Exported LaTeX tables to:', table_dir)
for name in generated_tables:
    print(' -', name)
